# ☕ CafePrompt: asistente de contenido con IA generativa para cafeterías de especialidad

- **Proyecto Final — Curso "IA: Generación de Prompts" (Coderhouse)**
- **Autor:** Sebastián Felipe Muñoz Rivera
- **Tipo de entrega:** POC (*proof of concept*) en Jupyter Notebook — modelos **texto-texto**, **texto-imagen** y (extra) **texto-audio**, con interfaz de usuario.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebakine/CafePrompt/blob/main/CafePrompt_POC.ipynb)

---

### Índice
1. Resumen
2. Introducción (nombre, problema, propuesta de solución, viabilidad)
3. Objetivos
4. Metodología
5. Herramientas y tecnologías (técnicas de *fast prompting*)
6. Implementación (código)
   - 6.1 Configuración
   - 6.2 Datos de entrada
   - 6.3 Prompts texto-texto (P0 a P5)
   - 6.4 Ejecución del *pipeline*
   - 6.5 Texto-imagen (NightCafe): prompts y resultados
   - 6.6 Texto-audio (gTTS) — extra
   - 6.7 Evaluación y optimización de prompts
   - 6.8 Interfaz de usuario (Gradio) — extra
7. Resultados
8. Conclusiones
9. Referencias

## 1. Resumen

Las cafeterías y tostadurías de especialidad pequeñas venden un producto cuyo valor depende de información técnica (origen, variedad, proceso, altitud, notas de cata, puntaje SCA) que el cliente promedio no entiende, y rara vez cuentan con presupuesto para un redactor, un fotógrafo o un *community manager*. Cada vez que llega un nuevo lote hay que producir desde cero la descripción para la carta, las publicaciones para redes sociales y las imágenes que lo acompañan; en la práctica esto se hace tarde, de forma inconsistente o simplemente no se hace.

**CafePrompt** es una prueba de concepto que convierte la **ficha técnica de un lote de café** en un **kit de contenido listo para usar** mediante una cadena de prompts optimizados: (1) una descripción sensorial comprensible para el cliente, (2) tres variantes de *copy* para Instagram, (3) un prompt de imagen construido automáticamente por el modelo de texto y ejecutado en **NightCafe** (texto-imagen), y (4) un guion locutado con **gTTS** (texto-audio). El texto se genera con la API gratuita de **Google Gemini**, aplicando *role prompting*, *few-shot*, salida estructurada en JSON, razonamiento guiado, encadenamiento de prompts y restricciones explícitas. La calidad se mide con controles automáticos y con una rúbrica de evaluación (*LLM-as-judge*) que compara un prompt ingenuo con el prompt optimizado. Todo el proyecto funciona con herramientas de costo cero.

## 2. Introducción

### 2.1 Nombre del proyecto
**CafePrompt** — asistente de contenido con IA generativa para cafeterías de especialidad.

### 2.2 Presentación del problema

**¿Cuál es el problema?** Una cafetería de especialidad rota lotes de café con frecuencia (microlotes que duran semanas). Por cada lote debe comunicar *por qué* ese café es distinto y vale más que un café comercial. La información disponible es la ficha del productor o importador, escrita en jerga técnica:

> *"Etiopía, Guji, Heirloom, natural, 2.000 msnm, 87 pts, arándano, frutilla, chocolate de leche."*

Para un cliente sin formación en cata, esta ficha no dice nada. Traducirla en un mensaje atractivo, fiel a los datos y consistente con la marca requiere tres habilidades que rara vez conviven en un equipo pequeño: **conocimiento sensorial**, **redacción publicitaria** y **producción visual**.

**¿Por qué es una problemática?**
- **Costo y tiempo:** contratar redacción, fotografía de producto y gestión de redes es inabordable para un negocio de 2 a 6 personas; el tiempo del barista se destina a la operación, no al marketing.
- **Brecha de comunicación:** si el cliente no entiende el valor del café, lo compara solo por precio y el negocio pierde su diferenciación.
- **Inconsistencia:** cada descripción la escribe una persona distinta, con otro tono, a veces con datos inventados ("notas a vainilla" que no están en la ficha) o con afirmaciones de salud que no corresponden.
- **Frecuencia:** el problema se repite con cada lote nuevo; no es una tarea única, sino un flujo recurrente que se beneficia de la automatización.

**¿Por qué es relevante resolverlo?** El café de especialidad compite por **experiencia e información**, no por volumen. Mejorar la comunicación de cada lote impacta directamente en la venta de granos y bebidas, en la educación del consumidor y en la trazabilidad hacia el productor. Una solución basada en prompts es replicable por cualquier cafetería sin conocimientos de programación.

### 2.3 Desarrollo de la propuesta de solución

La solución se vincula directamente con el uso de **modelos de IA generativa**: se usa un **modelo de lenguaje (texto-texto)** como "redactor catador" y un **modelo de difusión (texto-imagen)** como "fotógrafo de producto". El núcleo del proyecto es el **diseño y la optimización de prompts**, no el entrenamiento de modelos.

**Flujo (encadenamiento de prompts):**

```
Ficha técnica del lote (datos)
        │
        ▼
 P1  Texto-texto · Ficha sensorial para clientes (JSON)  ◄── few-shot + rol + restricciones
        │
        ├──► P2  Texto-texto · 3 variantes de copy para Instagram (JSON)
        │
        ├──► P3  Texto-texto · Meta-prompt → prompt de imagen optimizado (inglés)
        │            │
        │            ▼
        │        Texto-imagen · NightCafe genera la imagen del producto
        │
        ├──► P4  Texto-texto · Guion para locución (≈30 s)
        │            │
        │            ▼
        │        Texto-audio · gTTS genera el MP3            (extra)
        │
        └──► P5  Texto-texto · Evaluación con rúbrica (LLM-as-judge) vs. P0 (prompt ingenuo)
```

**Prompts de cada etapa:**

| Etapa | Prompt | Modelo | Qué resuelve |
|---|---|---|---|
| Línea base | **P0** — prompt ingenuo de una línea | texto-texto | Sirve como punto de comparación para medir la mejora |
| 1 | **P1** — ficha sensorial para cliente | texto-texto | Traduce la jerga técnica a lenguaje de cliente, en JSON reutilizable |
| 2 | **P2** — *copy* para Instagram | texto-texto | Tres enfoques (origen, sensorial, educativo) listos para publicar |
| 3 | **P3** — meta-prompt de imagen | texto-texto → texto-imagen | El LLM escribe el prompt visual a partir de las notas de cata |
| 4 | **P4** — guion de locución | texto-texto → texto-audio | Audio para Reels o para accesibilidad de la carta |
| 5 | **P5** — rúbrica de evaluación | texto-texto | Mide fidelidad, claridad, tono y formato |

### 2.4 Justificación de la viabilidad del proyecto

| Recurso | Elección | Justificación |
|---|---|---|
| Modelo texto-texto | **Google Gemini API** (capa gratuita, modelo *Flash-Lite*) | Sin costo, sin tarjeta de crédito, buena calidad en español, soporta salida JSON nativa. El *pipeline* completo usa ~20 llamadas, muy por debajo de los límites diarios de la capa gratuita. |
| Modelo texto-imagen | **NightCafe Studio** (créditos gratuitos diarios) | DALL·E dejó de ser gratuito. NightCafe es la alternativa sugerida por el curso: permite elegir modelo, relación de aspecto y *negative prompt*. Según la consigna, el prompt se escribe directamente en la herramienta y la imagen se agrega al repositorio. |
| Texto-audio (extra) | **gTTS** | Librería gratuita, sin clave, voz en español latinoamericano. |
| Interfaz (extra) | **Gradio** | Crea una UI web en pocas líneas y funciona dentro de Colab. |
| Entorno | **Google Colab** + GitHub | Gratuito, sin instalación local, reproducible desde el botón *Open in Colab*. |

**Viabilidad técnica y de tiempo:** el alcance está acotado a una POC de 3 lotes de café, lo que se puede construir y probar en pocas jornadas de trabajo. No se entrena ningún modelo; el esfuerzo se concentra en ingeniería de prompts.

**Riesgos y mitigaciones:**
- *Límites de uso de la capa gratuita (error 429)* → reintentos con espera progresiva y lista de modelos de respaldo.
- *Cambios de disponibilidad de modelos* → el modelo es un parámetro; el cliente prueba varios candidatos.
- *Ausencia de clave de API al revisar el notebook* → **modo caché**: las respuestas reales obtenidas en la ejecución original se guardan en `outputs/cache_respuestas.json` y el notebook se ejecuta completo igualmente.
- *Alucinaciones (datos inventados)* → restricciones explícitas ("usa solo la ficha"), controles automáticos y rúbrica de fidelidad.
- *Texto ilegible en imágenes generadas* → el prompt de imagen prohíbe texto y logos; la marca se agrega después en diseño.

## 3. Objetivos

**Objetivo general**
Desarrollar una prueba de concepto que, a partir de la ficha técnica de un lote de café de especialidad, genere automáticamente un kit de comunicación (texto, imagen y audio) fiel a los datos, comprensible para el cliente y alineado a la marca, usando exclusivamente técnicas de *prompting* y herramientas gratuitas.

**Objetivos específicos**
1. **Identificar** la problemática de comunicación de valor en cafeterías de especialidad pequeñas y plantear una solución con IA generativa (texto-texto y texto-imagen).
2. **Diseñar** una cadena de prompts (P1 a P4) que transforme datos técnicos en: descripción para la carta, *copy* para Instagram, prompt de imagen y guion de audio.
3. **Generar** imágenes de producto con una herramienta gratuita (NightCafe) a partir de prompts construidos por el modelo de texto.
4. **Optimizar** los prompts: comparar un prompt ingenuo (P0) contra el prompt optimizado (P1), y un prompt de imagen básico (v1) contra el optimizado (v2).
5. **Evaluar** los resultados con controles automáticos (formato, extensión, idioma) y una rúbrica cuantitativa (P5).
6. **Evaluar la disponibilidad de recursos:** operar con costo cero, midiendo tokens y número de llamadas.
7. *(Extra)* Integrar un tercer modelo (texto-audio) y una interfaz de usuario.

## 4. Metodología

El proyecto se desarrolla en **cinco fases**, siguiendo un ciclo iterativo *diseñar → probar → medir → ajustar* propio de la ingeniería de prompts:

| Fase | Procedimiento | Justificación |
|---|---|---|
| **1. Definición del problema y de los datos** | Se define una ficha técnica estándar (origen, región, variedad, proceso, altitud, notas, puntaje, tueste, método) y un perfil de marca (nombre, tono, público, restricciones). Se preparan 3 lotes contrastantes: Etiopía natural, Colombia lavado y Brasil *pulped natural*. | Datos estructurados reducen la ambigüedad del prompt y permiten verificar la fidelidad de la salida. Los tres lotes cubren perfiles sensoriales distintos (frutal, floral-cítrico, chocolatoso). |
| **2. Línea base** | Se ejecuta un prompt ingenuo (P0) sin rol, formato ni restricciones. | Sin línea base no es posible demostrar que la optimización mejora algo. |
| **3. Diseño de prompts optimizados** | Se construyen P1–P4 con la arquitectura **rol + contexto + tarea + formato de salida + restricciones**, y se encadenan (la salida JSON de P1 es entrada de P2, P3 y P4). | La estructura explícita y el JSON hacen la salida predecible y reutilizable; el encadenamiento divide un problema complejo en tareas simples. |
| **4. Generación multimodal** | El prompt de imagen producido por P3 se ejecuta manualmente en NightCafe (misma configuración para todos); el guion de P4 se convierte en audio con gTTS. | Conecta el modelo de texto con el de imagen: el LLM "traduce" notas de cata a lenguaje visual, algo difícil de hacer a mano. |
| **5. Evaluación y ajuste** | (a) Controles automáticos: JSON válido, límites de palabras, cantidad de *hashtags*, ausencia de voseo y de afirmaciones de salud; (b) rúbrica P5 de 1 a 5 en cinco criterios, comparando P0 vs. P1; (c) comparación visual imagen v1 vs. v2; (d) registro de tokens y latencia. | Combinar métricas deterministas con evaluación por rúbrica da una medida objetiva y otra cualitativa de la calidad. |

**Parámetros de generación:** temperatura baja (0,3) para las tareas que exigen fidelidad (P1, P3), media (0,8) para creatividad publicitaria (P2) y 0 para la evaluación (P5).

## 5. Herramientas y tecnologías

### 5.1 Técnicas de *fast prompting* utilizadas

| Técnica | Dónde se aplica | Justificación |
|---|---|---|
| **Role prompting** (asignación de rol) | *System instruction* común a P1–P4: "redactor senior de marketing gastronómico con formación de catador SCA" | Fija el vocabulario, el nivel de experticia y el tono sin repetirlos en cada prompt. |
| **Zero-shot** | P0 (línea base), P2 y P4 | Para tareas que el modelo ya domina (redactar un *post*), basta con instrucciones claras; ahorra tokens. |
| **Few-shot** (aprendizaje con ejemplos) | P1: se entrega un ejemplo completo *ficha → JSON* con un café de Kenia | Enseña el formato, la extensión y el estilo de las analogías sensoriales mejor que cualquier descripción. |
| **Salida estructurada (JSON)** | P1, P2, P3 y P5 (`response_mime_type="application/json"`) | Permite encadenar prompts, validar automáticamente y reutilizar los campos en la carta, la web o la UI. |
| **Delimitadores** | Las fichas y los datos van entre comillas triples y encabezados `### ... ###` | Separan instrucciones de datos y reducen el riesgo de que el modelo confunda ambos. |
| **Restricciones explícitas y prompts negativos** | "Usa solo los datos de la ficha", "no hagas afirmaciones de salud", "sin voseo", "sin texto ni logos en la imagen", límites de palabras | Controlan las alucinaciones, el tono regional y los errores típicos de los modelos de imagen. |
| **Razonamiento guiado (*chain-of-thought* estructurado)** | P3: el modelo primero llena `analisis_visual` (nota → color/textura/objeto) y luego redacta el prompt de imagen | Descomponer el razonamiento mejora la coherencia entre las notas de cata y la imagen. |
| **Encadenamiento de prompts (*prompt chaining*)** | P1 → P2 / P3 / P4 → P5 | Divide la tarea en pasos verificables; cada prompt es corto y especializado. |
| **Meta-prompting** | P3: un LLM escribe el prompt para otro modelo (texto-imagen) | Automatiza la parte más difícil del *prompting* visual y lo hace consistente entre lotes. |
| **Plantillas parametrizadas** | Todos los prompts se construyen con `str.format()` sobre la ficha | Hace la solución reutilizable para cualquier lote nuevo. |
| **LLM-as-judge con rúbrica** | P5 | Evaluación reproducible (temperatura 0) de la mejora entre P0 y P1. |
| **Estructura de prompt visual** | Imágenes v2: sujeto + elementos + ambientación + composición + luz + estilo + paleta + calidad + *negative prompt* | Es la estructura recomendada para modelos de difusión; la comparación v1 vs. v2 muestra el efecto. |

### 5.2 Tecnologías

- **Python 3 / Jupyter / Google Colab** — entorno de la POC.
- **`google-genai`** — SDK oficial de la API de Gemini (texto-texto).
- **NightCafe Studio** — generación de imágenes (texto-imagen), uso manual según la consigna.
- **`gTTS`** — síntesis de voz (texto-audio).
- **`gradio`** — interfaz de usuario.
- **`pandas`** — tablas de evaluación y uso de tokens.

## 6. Implementación

### 6.1 Configuración del entorno

**Cómo ejecutar:**
1. Abre el notebook en Colab con el botón *Open in Colab*.
2. *(Opcional, para generar respuestas nuevas)* Crea una clave gratuita en [Google AI Studio](https://aistudio.google.com/apikey) y agrégala en Colab → 🔑 **Secrets** con el nombre `GEMINI_API_KEY` (activa el acceso para este notebook).
3. `Entorno de ejecución → Ejecutar todas`.

Sin clave, el notebook funciona en **modo caché**: reutiliza las respuestas reales de Gemini guardadas en `outputs/cache_respuestas.json` durante la ejecución original.

In [ ]:
# 6.1.a Instalación de dependencias (en Colab tarda ~1 minuto)
import subprocess, sys
paquetes = ["google-genai", "gTTS", "gradio"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *paquetes],
                   capture_output=True, text=True)
print("Dependencias instaladas ✅" if r.returncode == 0 else r.stderr[-800:])

In [ ]:
# 6.1.b Si se ejecuta en Colab, se clona el repositorio para disponer de imágenes, caché y audio
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/sebakine/CafePrompt.git"
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB and not Path("prompts").exists():
    if not Path("CafePrompt").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("CafePrompt")
if EN_COLAB:   # sincroniza con la última versión publicada del repositorio
    subprocess.run("git fetch -q origin && git reset -q --hard origin/main", shell=True)

print("¿Ejecutando en Colab?:", EN_COLAB)
print("Directorio de trabajo:", os.getcwd())

In [ ]:
# 6.1.c Configuración general, clave de API y modo de ejecución
import json, re, time, textwrap, hashlib, logging
import pandas as pd
logging.getLogger("google_genai").setLevel(logging.ERROR)   # oculta avisos internos del SDK
from IPython.display import display, Markdown, Image, Audio

DIR_OUT   = Path("outputs")
DIR_AUDIO = DIR_OUT / "audio"
DIR_IMG   = Path("images")
CACHE_PATH = DIR_OUT / "cache_respuestas.json"
for d in (DIR_OUT, DIR_AUDIO, DIR_IMG):
    d.mkdir(parents=True, exist_ok=True)

# Modelos candidatos de la capa gratuita, en orden de preferencia (se usa el primero disponible)
MODELOS_CANDIDATOS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite",
                      "gemini-2.5-flash-lite", "gemini-2.5-flash"]

def obtener_api_key():
    # Nunca se escribe la clave en el código: se lee de variable de entorno o de Colab Secrets
    key = os.environ.get("GEMINI_API_KEY")
    if not key and EN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception as e:
            print("Aviso: no se pudo leer el secreto GEMINI_API_KEY →", type(e).__name__)
    return key

API_KEY = obtener_api_key()
MODO = "api" if API_KEY else "cache"
# Con REGENERAR=False, si un prompt idéntico ya tiene respuesta en caché se reutiliza (ahorra cuota y
# mantiene la coherencia entre los prompts de imagen publicados y las imágenes generadas en NightCafe).
REGENERAR = False
print(f"Modo de ejecución: {MODO.upper()}",
      "(respuestas nuevas desde Gemini)" if MODO == "api" else "(respuestas reales guardadas en caché)")

In [ ]:
# 6.1.d Cliente LLM con reintentos, modelos de respaldo, registro de tokens y caché
CACHE = json.loads(CACHE_PATH.read_text(encoding="utf-8")) if CACHE_PATH.exists() else {}
REGISTRO_USO = []   # una fila por llamada: clave, modelo, tokens, latencia

class ClienteLLM:
    def __init__(self, api_key, modelos):
        self.modelos, self.modelo = modelos, None
        self.client = None
        if api_key:
            from google import genai
            self.client = genai.Client(api_key=api_key)

    def generar(self, clave, prompt, system=None, temperatura=0.4, json_mode=False,
                max_tokens=4096, guardar=True):
        # Reutiliza la respuesta real registrada si el prompt, el rol y la temperatura son idénticos
        item = CACHE.get(clave)
        vigente = bool(item) and item.get("prompt") == prompt and item.get("system") == system \
                  and item.get("temperatura") == temperatura
        if vigente and (self.client is None or not REGENERAR):
            REGISTRO_USO.append({"clave": clave, "modelo": item.get("modelo"), "origen": "caché", **item.get("uso", {})})
            return item["respuesta"]
        if self.client is None:
            raise KeyError(f"'{clave}' no está en caché (o el prompt cambió); configura GEMINI_API_KEY.")

        from google.genai import types
        cfg = types.GenerateContentConfig(
            system_instruction=system, temperature=temperatura, max_output_tokens=max_tokens,
            response_mime_type="application/json" if json_mode else "text/plain")
        ultimo_error = None
        for modelo in ([self.modelo] if self.modelo else self.modelos):
            for intento in range(4):
                try:
                    t0 = time.time()
                    r = self.client.models.generate_content(model=modelo, contents=prompt, config=cfg)
                    texto = (r.text or "").strip()
                    if not texto:
                        raise ValueError("Respuesta vacía")
                    um = r.usage_metadata
                    uso = {"tokens_entrada": getattr(um, "prompt_token_count", None),
                           "tokens_salida": getattr(um, "candidates_token_count", None),
                           "tokens_total": getattr(um, "total_token_count", None),
                           "latencia_s": round(time.time() - t0, 2)}
                    self.modelo = modelo
                    REGISTRO_USO.append({"clave": clave, "modelo": modelo, "origen": "API", **uso})
                    if guardar:
                        CACHE[clave] = {"modelo": modelo, "temperatura": temperatura,
                                        "system": system, "prompt": prompt,
                                        "respuesta": texto, "uso": uso}
                        CACHE_PATH.write_text(json.dumps(CACHE, ensure_ascii=False, indent=2),
                                              encoding="utf-8")
                    return texto
                except Exception as e:
                    ultimo_error, msg = e, str(e)
                    if any(c in msg for c in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "vacía")):
                        espera = 20 * (intento + 1)
                        print(f"  ↻ {modelo}: límite/indisponibilidad, reintento en {espera}s")
                        time.sleep(espera)
                        continue
                    print(f"  ✗ {modelo} no disponible ({type(e).__name__}); probando el siguiente…")
                    break
        raise RuntimeError(f"No fue posible generar '{clave}': {ultimo_error}")

def a_json(texto):
    # Convierte la respuesta en dict tolerando bloques ```json
    limpio = re.sub(r"^```(?:json)?|```$", "", texto.strip(), flags=re.M).strip()
    return json.loads(limpio)

llm = ClienteLLM(API_KEY, MODELOS_CANDIDATOS)
print("Cliente listo. Respuestas en caché:", len(CACHE))

### 6.2 Datos de entrada: perfil de marca y fichas técnicas

Se usa una marca ficticia (**Café Altura**, tostaduría de especialidad en Santiago) y tres lotes con perfiles sensoriales contrastantes. Los datos son representativos de fichas reales de importadores, pero los nombres de fincas son ficticios.

In [ ]:
MARCA = {
    "nombre": "Café Altura",
    "tipo": "tostaduría y cafetería de especialidad en Santiago de Chile",
    "tono": "cercano, experto y cálido; educa sin sonar técnico; español de Chile neutro, sin modismos",
    "publico": "adultos de 25 a 45 años curiosos por el café, que aún no dominan términos de cata",
    "prohibido": "voseo o modismos argentinos, garabatos, afirmaciones de salud, superlativos absolutos ('el mejor del mundo'), datos no presentes en la ficha",
}

LOTES = [
    {"id": "etiopia", "pais": "Etiopía", "region": "Guji, Oromía", "productor": "Estación de lavado Hambela (cooperativa de pequeños productores)",
     "variedad": "Heirloom etíope (variedades locales)", "proceso": "Natural (secado en camas africanas por 21 días)",
     "altitud": "1.950–2.100 msnm", "notas": "arándano, frutilla madura, chocolate de leche, jazmín",
     "acidez_cuerpo": "acidez brillante tipo frutos rojos, cuerpo medio y sedoso", "puntaje_sca": 87,
     "tueste": "claro", "metodo_sugerido": "V60 o Chemex", "precio": "$12.900 (250 g)"},
    {"id": "colombia", "pais": "Colombia", "region": "Pitalito, Huila", "productor": "Finca La Esperanza (familia Rojas)",
     "variedad": "Pink Bourbon", "proceso": "Lavado con fermentación prolongada de 36 horas",
     "altitud": "1.750 msnm", "notas": "mandarina, panela, flor de azahar, té negro",
     "acidez_cuerpo": "acidez cítrica jugosa, cuerpo ligero y limpio", "puntaje_sca": 86.5,
     "tueste": "claro-medio", "metodo_sugerido": "V60, AeroPress o espresso", "precio": "$11.900 (250 g)"},
    {"id": "brasil", "pais": "Brasil", "region": "Cerrado Mineiro", "productor": "Fazenda Santa Clara",
     "variedad": "Catuaí Amarillo", "proceso": "Pulped natural (honey)",
     "altitud": "1.100 msnm", "notas": "avellana tostada, cacao, caramelo, ciruela seca",
     "acidez_cuerpo": "acidez baja, cuerpo alto y cremoso", "puntaje_sca": 84,
     "tueste": "medio", "metodo_sugerido": "espresso, flat white o prensa francesa", "precio": "$9.900 (250 g)"},
]

ETIQUETAS = {"pais": "País", "region": "Región", "productor": "Productor", "variedad": "Variedad",
             "proceso": "Proceso", "altitud": "Altitud", "notas": "Notas de cata",
             "acidez_cuerpo": "Acidez y cuerpo", "puntaje_sca": "Puntaje SCA", "tueste": "Tueste",
             "metodo_sugerido": "Método sugerido", "precio": "Precio"}

def ficha_a_texto(lote):
    return "\n".join(f"- {ETIQUETAS[k]}: {v}" for k, v in lote.items() if k in ETIQUETAS)

display(pd.DataFrame(LOTES).set_index("id").T)

### 6.3 Diseño de prompts texto-texto

Todos los prompts optimizados comparten una **instrucción de sistema (rol)** y siguen la arquitectura **rol → contexto → tarea → formato de salida → restricciones**. Los datos van siempre entre delimitadores.

#### P0 — Línea base (prompt ingenuo, *zero-shot* sin estructura)
Es el prompt que escribiría cualquier persona sin conocimientos de *prompting*. Se usa solo para comparar.

In [ ]:
P0_BASELINE = "Describe este café para venderlo en mi cafetería: {pais}, {region}, {variedad}, {proceso}, {altitud}, notas: {notas}, {puntaje_sca} puntos."

#### Instrucción de sistema común (*role prompting*)

In [ ]:
SYSTEM_ROL = textwrap.dedent(f"""
Eres «CaféPrompt», redactor senior de marketing gastronómico especializado en café de especialidad,
con formación de catador (protocolo SCA) y diez años de experiencia en tostadurías de Latinoamérica.
Trabajas para {MARCA['nombre']}, {MARCA['tipo']}.
Tono de marca: {MARCA['tono']}.
Público: {MARCA['publico']}.
Reglas permanentes:
1. Usa ÚNICAMENTE los datos de la ficha técnica entregada; si un dato no está, omítelo. Nunca inventes notas, premios ni cifras.
2. Traduce la jerga técnica (proceso, altitud, variedad) a experiencias sensoriales que un cliente entienda.
3. Está prohibido: {MARCA['prohibido']}.
4. Escribe en español de Chile neutro, con tuteo ("prueba", "descubre"), nunca voseo.
""").strip()
print(SYSTEM_ROL)

#### P1 — Ficha sensorial para clientes (*few-shot* + salida JSON + restricciones)
Técnicas: rol, *few-shot* (un ejemplo completo con un café de Kenia que **no** forma parte de los lotes), salida estructurada, delimitadores y límites de extensión.

In [ ]:
EJEMPLO_FICHA = """- País: Kenia
- Región: Nyeri
- Variedad: SL28 y SL34
- Proceso: Lavado
- Altitud: 1.800 msnm
- Notas de cata: grosella negra, pomelo, panela
- Acidez y cuerpo: acidez intensa y jugosa, cuerpo medio
- Puntaje SCA: 88
- Tueste: claro
- Método sugerido: V60"""

EJEMPLO_SALIDA = {
    "nombre_comercial": "Kenia Nyeri · Grosella y Pomelo",
    "descripcion_menu": "Un café vibrante y jugoso: recuerda a un jugo de grosellas con un toque de pomelo y un final dulce de panela. Ideal para quienes buscan una taza fresca y llena de fruta.",
    "descripcion_extendida": "Cultivado a 1.800 metros en las laderas de Nyeri, este lote de variedades SL28 y SL34 pasó por un proceso lavado que deja su sabor muy limpio y definido. En la taza encontrarás una acidez intensa, parecida a la de una fruta recién cortada, que recuerda a la grosella negra y al pomelo, y que termina con un dulzor de panela. Con 88 puntos en la escala SCA, es un café para descubrir sin azúcar y apreciar cómo cambia a medida que se enfría.",
    "notas_para_cliente": [
        {"nota": "grosella negra", "analogia": "como un jugo de berries poco dulce"},
        {"nota": "pomelo", "analogia": "frescura cítrica, similar a la cáscara del pomelo"},
        {"nota": "panela", "analogia": "dulzor parecido a la chancaca"}],
    "intensidad_1a5": 3,
    "acidez_1a5": 5,
    "preparacion_recomendada": "En V60, 15 g de café para 250 ml de agua a 93 °C; tiempo total cercano a 3 minutos.",
    "maridaje": "Kuchen de frambuesa",
    "ideal_para": "Quien disfruta los sabores frutales y quiere salir del café amargo tradicional."
}

P1_FICHA_CLIENTE = """### TAREA ###
Transforma la ficha técnica de un lote de café en una ficha sensorial para clientes de la cafetería.

### FORMATO DE SALIDA ###
Responde solo con un objeto JSON con estas claves exactas:
nombre_comercial (máx. 6 palabras), descripcion_menu (2 oraciones, entre 30 y 45 palabras),
descripcion_extendida (5 o 6 oraciones, entre 85 y 115 palabras: origen → proceso → taza → invitación),
notas_para_cliente (lista con una entrada por cada nota de cata: {{"nota", "analogia"}}; analogías con alimentos conocidos en Chile),
intensidad_1a5 (entero), acidez_1a5 (entero), preparacion_recomendada (una oración con receta),
maridaje (un alimento o pastelería habitual en Chile), ideal_para (una oración).

### EJEMPLO ###
Ficha:
\"\"\"
{ejemplo_ficha}
\"\"\"
Salida:
{ejemplo_salida}

### FICHA A TRANSFORMAR ###
\"\"\"
{ficha}
\"\"\"
Salida:"""

#### P2 — *Copy* para Instagram (*zero-shot* con restricciones + encadenamiento)
Recibe la salida JSON de P1 (no la ficha técnica), lo que garantiza coherencia entre carta y redes.

In [ ]:
P2_COPY_INSTAGRAM = """### CONTEXTO ###
La cafetería lanzará este café esta semana. Esta es su ficha sensorial aprobada:
\"\"\"
{ficha_cliente}
\"\"\"

### TAREA ###
Escribe 3 publicaciones distintas para el feed de Instagram, una por enfoque: "origen", "sensorial" y "educativo".

### FORMATO DE SALIDA ###
JSON con la clave "variantes": lista de 3 objetos con claves
enfoque, gancho (primera línea, máx. 10 palabras), cuerpo (50 a 110 palabras), llamado_a_la_accion (una oración), hashtags (lista de 5 a 8, sin espacios).

### RESTRICCIONES ###
- Máximo 3 emojis por publicación.
- Menciona el precio solo en una de las tres variantes: {precio}.
- No uses información que no esté en la ficha sensorial.
- Hashtags en español y en formato CamelCase, sin tildes ni duplicados (ej.: #CafeDeEspecialidad), salvo #specialtycoffee.
- Reescribe con palabras propias: no copies oraciones textuales de la ficha sensorial.
- Cada variante debe abrir con un gancho distinto (pregunta, dato de origen o invitación)."""

#### P3 — Meta-prompt: el LLM construye el prompt de imagen (razonamiento guiado + plantilla visual)
Conecta el modelo texto-texto con el texto-imagen. Primero obliga al modelo a **razonar** en un campo explícito (qué color, textura u objeto representa cada nota) y luego a **componer** el prompt con una estructura fija. El prompt final se redacta en inglés porque los modelos de difusión disponibles en NightCafe fueron entrenados mayoritariamente con descripciones en ese idioma.

In [ ]:
P3_META_PROMPT_IMAGEN = """### CONTEXTO ###
Necesitamos una fotografía publicitaria de producto para Instagram (formato vertical 4:5) de este café:
\"\"\"
{ficha_tecnica}
\"\"\"

### TAREA (sigue los pasos en orden) ###
Paso 1 — En "analisis_visual", asocia cada nota de cata a un elemento visual concreto (fruta, flor, ingrediente, color, textura)
y el origen/proceso a una ambientación sutil (materiales, paisaje, luz). Justifica cada asociación en pocas palabras.
Paso 2 — En "prompt_imagen", redacta en INGLÉS un prompt de 55 a 75 palabras (frases cortas separadas por comas) con este orden:
[sujeto principal: taza y bolsa de café sin etiqueta] + [elementos de las notas] + [ambientación del origen]
+ [composición y encuadre] + [iluminación] + [estilo fotográfico y lente] + [paleta de colores] + [calidad].
Paso 3 — En "prompt_negativo", lista en inglés lo que se debe evitar.

### FORMATO DE SALIDA ###
JSON con claves: analisis_visual (lista de objetos {{"elemento_ficha", "representacion_visual", "motivo"}}),
prompt_imagen (string), prompt_negativo (string), relacion_aspecto ("4:5").

### RESTRICCIONES ###
- La imagen NO debe contener texto, letras, logos ni marcas (los modelos de imagen los deforman; la marca se agrega después).
  Estas exclusiones van SOLO en "prompt_negativo": no escribas "text", "logo" ni "label" en el prompt positivo,
  porque mencionarlos puede inducir al modelo a dibujarlos. Describe la bolsa como "plain unbranded".
- Estética natural y realista, no caricaturesca. Sin personas.
- No incluyas elementos que contradigan las notas de cata."""

#### P4 — Guion para locución (texto → audio)
Optimizado para síntesis de voz: frases cortas, números escritos en palabras, sin emojis ni símbolos que el TTS leería mal.

In [ ]:
P4_GUION_AUDIO = """### CONTEXTO ###
Ficha sensorial del café:
\"\"\"
{ficha_cliente}
\"\"\"

### TAREA ###
Escribe el guion de una locución de unos 30 segundos (5 o 6 oraciones, entre 60 y 80 palabras) para un Reel de Instagram
y para la versión accesible de la carta (personas con discapacidad visual).

### RESTRICCIONES ###
- Texto plano, sin emojis, hashtags, viñetas, comillas ni símbolos.
- Frases cortas y naturales para ser leídas por una voz sintética.
- Escribe los números en palabras (por ejemplo, "mil novecientos metros").
- Termina invitando a probarlo en {marca}."""

#### P5 — Evaluación con rúbrica (*LLM-as-judge*, temperatura 0)
Compara la salida del prompt ingenuo (P0) con la descripción extendida del prompt optimizado (P1). Las etiquetas "A" y "B" ocultan cuál es cuál para reducir sesgos.

In [ ]:
P5_RUBRICA = """### ROL ###
Actúa como evaluador imparcial de contenidos de marketing de café de especialidad.

### FICHA TÉCNICA (fuente de verdad) ###
\"\"\"
{ficha_tecnica}
\"\"\"

### TEXTO A ###
\"\"\"
{texto_a}
\"\"\"

### TEXTO B ###
\"\"\"
{texto_b}
\"\"\"

### CONTEXTO DE USO ###
Cada texto debe servir como descripción del café en la web y en la carta de la cafetería y se publicará TAL CUAL:
no puede requerir que el dueño elija entre opciones, borre instrucciones, quite formato markdown o recorte extensión
(máximo recomendado: 120 palabras).

### TAREA ###
Evalúa cada texto de 1 (muy deficiente) a 5 (excelente) en:
fidelidad (no inventa ni omite datos clave de la ficha; cualquier método, sabor o dato ausente en la ficha es un dato inventado),
claridad (lo entiende alguien sin formación en cata), tono_marca (cercano, experto, español de Chile neutro, sin voseo),
persuasion (motiva la compra), uso_directo (se puede publicar tal cual según el contexto de uso).
Lista además los datos inventados que detectes en cada texto. Sé estricto y justifica con evidencia.

### FORMATO DE SALIDA ###
JSON: {{"A": {{"fidelidad": int, "claridad": int, "tono_marca": int, "persuasion": int, "uso_directo": int, "datos_inventados": [str]}},
        "B": {{...mismas claves...}}, "comentario": "máx. 40 palabras"}}"""

### 6.4 Ejecución del *pipeline* texto-texto

La función `ejecutar_pipeline()` encadena P0 → P1 → (P2, P3, P4) para un lote. Es la misma función que usa la interfaz de usuario.

In [ ]:
def ejecutar_pipeline(lote, prefijo=None, guardar=True):
    """Ejecuta la cadena de prompts para un lote y devuelve un dict con todas las salidas."""
    p = prefijo or lote["id"]
    ficha = ficha_a_texto(lote)
    out = {"lote": lote}

    # P0 — línea base (sin rol ni formato)
    out["p0_baseline"] = llm.generar(f"{p}/p0", P0_BASELINE.format(**lote),
                                     temperatura=0.7, guardar=guardar)

    # P1 — ficha sensorial (few-shot + JSON)
    prompt_p1 = P1_FICHA_CLIENTE.format(ejemplo_ficha=EJEMPLO_FICHA,
                                        ejemplo_salida=json.dumps(EJEMPLO_SALIDA, ensure_ascii=False, indent=2),
                                        ficha=ficha)
    out["p1_ficha"] = a_json(llm.generar(f"{p}/p1", prompt_p1, system=SYSTEM_ROL,
                                         temperatura=0.3, json_mode=True, guardar=guardar))
    ficha_cliente = json.dumps(out["p1_ficha"], ensure_ascii=False, indent=2)

    # P2 — copy Instagram (encadenado sobre P1)
    out["p2_copy"] = a_json(llm.generar(f"{p}/p2", P2_COPY_INSTAGRAM.format(ficha_cliente=ficha_cliente, precio=lote.get("precio", "")),
                                        system=SYSTEM_ROL, temperatura=0.8, json_mode=True, guardar=guardar))

    # P3 — meta-prompt de imagen (razonamiento guiado)
    out["p3_imagen"] = a_json(llm.generar(f"{p}/p3", P3_META_PROMPT_IMAGEN.format(ficha_tecnica=ficha),
                                          system=SYSTEM_ROL, temperatura=0.3, json_mode=True, guardar=guardar))

    # P4 — guion de audio
    out["p4_guion"] = llm.generar(f"{p}/p4", P4_GUION_AUDIO.format(ficha_cliente=ficha_cliente, marca=MARCA["nombre"]),
                                  system=SYSTEM_ROL, temperatura=0.5, guardar=guardar)
    return out

RESULTADOS = {}
for lote in LOTES:
    print(f"▶ Procesando lote: {lote['pais']} ({lote['id']})")
    RESULTADOS[lote["id"]] = ejecutar_pipeline(lote)
print("\nPipeline completado para", len(RESULTADOS), "lotes ✅")

#### Resultado P0 vs. P1 — descripción para el cliente

In [ ]:
def mostrar_descripciones(lid):
    r = RESULTADOS[lid]; f = r["p1_ficha"]
    notas = "\n".join(f"  - **{n['nota']}** → {n['analogia']}" for n in f["notas_para_cliente"])
    display(Markdown(f"""
---
## {r['lote']['pais']} — {f['nombre_comercial']}

**🔴 P0 · Prompt ingenuo** ({len(r['p0_baseline'].split())} palabras):

> {r['p0_baseline'][:1500].replace(chr(10), chr(10) + '> ')}

**🟢 P1 · Prompt optimizado**

- **Carta (≤45 palabras):** {f['descripcion_menu']}
- **Descripción extendida:** {f['descripcion_extendida']}
- **Notas explicadas:**
{notas}
- **Intensidad:** {f['intensidad_1a5']}/5 · **Acidez:** {f['acidez_1a5']}/5
- **Preparación:** {f['preparacion_recomendada']}
- **Maridaje:** {f['maridaje']} · **Ideal para:** {f['ideal_para']}
"""))

for lid in RESULTADOS:
    mostrar_descripciones(lid)

#### Resultado P2 — *copy* para Instagram

In [ ]:
for lid, r in RESULTADOS.items():
    display(Markdown(f"### {r['lote']['pais']}"))
    for v in r["p2_copy"]["variantes"]:
        display(Markdown(f"""**Enfoque {v['enfoque']}**
> **{v['gancho']}**
> {v['cuerpo']}
> *{v['llamado_a_la_accion']}*
> {' '.join(v['hashtags'])}
"""))

#### Resultado P3 — prompts de imagen construidos por el modelo de texto
Estos prompts son los que se copian en NightCafe (sección 6.5). También se muestra el razonamiento visual del modelo.

In [ ]:
for lid, r in RESULTADOS.items():
    p3 = r["p3_imagen"]
    display(Markdown(f"### {r['lote']['pais']}"))
    display(pd.DataFrame(p3["analisis_visual"]))
    display(Markdown(f"**Prompt de imagen (v2, optimizado):**\n```text\n{p3['prompt_imagen']}\n```\n"
                     f"**Negative prompt:**\n```text\n{p3['prompt_negativo']}\n```\n"
                     f"**Relación de aspecto:** {p3['relacion_aspecto']}"))

### 6.5 Texto-imagen con NightCafe Studio

**Herramienta:** [NightCafe Studio](https://creator.nightcafe.studio) (plan gratuito con créditos diarios).
Como no se usa DALL·E, **no se utiliza API**: siguiendo la consigna, cada prompt se escribió directamente en la herramienta y la imagen resultante se guardó en la carpeta `images/` del repositorio. Todos los prompts, parámetros e imágenes también están documentados en [`prompts/prompts_imagen.md`](prompts/prompts_imagen.md).

**Configuración común:** misma herramienta, mismo modelo y misma relación de aspecto (4:5 o la más cercana disponible) para que la comparación sea justa.

#### Experimento de optimización del prompt de imagen (lote Etiopía)
- **v1 — prompt básico:** lo que escribiría una persona sin técnica.
- **v2 — prompt optimizado:** generado por P3 (estructura sujeto + notas + ambientación + composición + luz + estilo + paleta + calidad, con *negative prompt*).

In [ ]:
PROMPT_IMAGEN_V1 = "a cup of coffee from Ethiopia"

IMAGENES = [
    {"archivo": "01_etiopia_v1_prompt_basico.jpg", "lote": "etiopia", "version": "v1 · prompt básico",
     "prompt": PROMPT_IMAGEN_V1, "negativo": "(ninguno)"},
    {"archivo": "02_etiopia_v2_prompt_optimizado.jpg", "lote": "etiopia", "version": "v2 · prompt optimizado (P3)"},
    {"archivo": "03_colombia_v2_prompt_optimizado.jpg", "lote": "colombia", "version": "v2 · prompt optimizado (P3)"},
    {"archivo": "04_brasil_v2_prompt_optimizado.jpg", "lote": "brasil", "version": "v2 · prompt optimizado (P3)"},
]
for img in IMAGENES:
    if "prompt" not in img:   # las versiones v2 usan exactamente el prompt generado por P3
        img["prompt"] = RESULTADOS[img["lote"]]["p3_imagen"]["prompt_imagen"]
        img["negativo"] = RESULTADOS[img["lote"]]["p3_imagen"]["prompt_negativo"]

for img in IMAGENES:
    ruta = DIR_IMG / img["archivo"]
    display(Markdown(f"#### {img['lote'].capitalize()} — {img['version']}\n"
                     f"**Prompt:** `{img['prompt']}`\n\n**Negative prompt:** `{img['negativo']}`"))
    if ruta.exists():
        display(Image(filename=str(ruta), width=420))
    else:
        print(f"(Imagen pendiente: {ruta})")

### 6.6 Texto-audio con gTTS *(contenido adicional)*

El guion generado por P4 se sintetiza con **gTTS** (Google Text-to-Speech, gratuito). Se usa el acento de español latinoamericano (`tld="com.mx"`), el más cercano disponible al español de Chile neutro.

In [ ]:
def texto_a_audio(texto, ruta):
    from gtts import gTTS
    tmp = Path(str(ruta) + ".tmp")
    try:
        gTTS(text=texto, lang="es", tld="com.mx").save(str(tmp))
        tmp.replace(ruta)          # solo se reemplaza si la síntesis terminó bien
    finally:
        tmp.unlink(missing_ok=True)
    return ruta

for lid, r in RESULTADOS.items():
    guion = r["p4_guion"]
    ruta = DIR_AUDIO / f"guion_{lid}.mp3"
    display(Markdown(f"#### {r['lote']['pais']} — guion ({len(guion.split())} palabras)\n> {guion}"))
    try:
        if MODO == "api" or not ruta.exists():
            texto_a_audio(guion, ruta)
    except Exception as e:
        print("No se pudo sintetizar el audio (¿sin internet?):", type(e).__name__)
    if ruta.exists() and ruta.stat().st_size > 0:
        display(Audio(filename=str(ruta)))

### 6.7 Evaluación y optimización de prompts

#### 6.7.a Controles automáticos (deterministas)
Se verifica que las salidas cumplan las restricciones pedidas en los prompts.

In [ ]:
VOSEO = re.compile(r"\b(tenés|probá|querés|podés|sabés|vení|mirá|descubrí|disfrutá|vos)\b", re.I)
SALUD = re.compile(r"\b(saludable|antioxidante|adelgaza|cura|previene|beneficios para la salud)\b", re.I)
n_palabras = lambda t: len(re.findall(r"\b\w+\b", t))

def hashtags_ok(tags):
    limpios = [t.lower() for t in tags]
    return (5 <= len(tags) <= 8 and len(set(limpios)) == len(limpios)
            and not any(re.search(r"[áéíóúñ]", t, re.I) for t in tags))

def evaluar_controles(resultados):
    filas = []
    for lid, r in resultados.items():
        f, c, g, p3 = r["p1_ficha"], r["p2_copy"]["variantes"], r["p4_guion"], r["p3_imagen"]
        todo_texto = json.dumps([f, c, g], ensure_ascii=False)
        filas.append({
            "lote": lid,
            "P1 carta 30-45 palabras": 30 <= n_palabras(f["descripcion_menu"]) <= 45,
            "P1 extendida 80-120": 80 <= n_palabras(f["descripcion_extendida"]) <= 120,
            "P1 cubre todas las notas": len(f["notas_para_cliente"]) >= len(r["lote"]["notas"].split(",")),
            "P2 3 variantes": len(c) == 3,
            "P2 hashtags 5-8, sin tildes ni duplicados": all(hashtags_ok(v["hashtags"]) for v in c),
            "P3 prompt ≤ 90 palabras": n_palabras(p3["prompt_imagen"]) <= 90,
            "P3 sin 'text/logo' en prompt positivo": not re.search(r"\b(text|logo|logos|label|labels|typography)\b", p3["prompt_imagen"], re.I),
            "P4 guion 60-85 palabras": 60 <= n_palabras(g) <= 85,
            "Sin voseo": not VOSEO.search(todo_texto),
            "Sin afirmaciones de salud": not SALUD.search(todo_texto),
        })
    return pd.DataFrame(filas).set_index("lote")

controles = evaluar_controles(RESULTADOS)
display(controles.apply(lambda col: col.map({True: "✅", False: "❌"})))
tasa = controles.values.mean() * 100
print(f"Cumplimiento global de restricciones (prompts v2): {tasa:.1f}%")

#### 6.7.b Métricas objetivas: prompt ingenuo (P0) vs. prompt optimizado (P1)
Se cuentan las palabras, los **métodos de preparación mencionados que no están en la ficha** (dato inventado verificable) y si el texto requiere edición manual (varias opciones, formato markdown o instrucciones al dueño).

In [ ]:
METODOS = ["v60", "chemex", "aeropress", "espresso", "expreso", "prensa francesa", "moka", "cold brew",
           "kalita", "sifón", "cafetera italiana", "clever"]

def metodos_inventados(texto, lote):
    t, permitidos = texto.lower(), lote["metodo_sugerido"].lower()
    return sorted({m for m in METODOS if m in t and m not in permitidos})

def requiere_edicion(texto):
    return bool(re.search(r"(opci[oó]n\s*\d|###|\*\*|^>|^---|aqu[ií] tienes)", texto, re.I | re.M))

objetivas = []
for lid, r in RESULTADOS.items():
    for nombre, texto in (("P0 ingenuo", r["p0_baseline"]), ("P1 optimizado", r["p1_ficha"]["descripcion_extendida"])):
        inv = metodos_inventados(texto, r["lote"])
        objetivas.append({"lote": lid, "prompt": nombre, "palabras": n_palabras(texto),
                          "métodos inventados": ", ".join(inv) or "—", "n_inventados": len(inv),
                          "requiere edición manual": requiere_edicion(texto)})
objetivas = pd.DataFrame(objetivas)
display(objetivas)
display(objetivas.groupby("prompt")[["palabras", "n_inventados", "requiere edición manual"]].mean().round(2))

#### 6.7.c Optimización iterativa de prompts: iteración 1 → iteración 2
La primera ejecución (iteración 1, guardada en `outputs/iteracion_1/`) reveló fallas concretas: descripciones extendidas bajo el mínimo de palabras, prompts de imagen demasiado largos y con las palabras *"no text, no logos"* dentro del prompt positivo (lo que puede inducir al modelo de imagen a dibujar texto), guiones de audio sobre el límite y *hashtags* con tildes o duplicados. Los cambios aplicados en la iteración 2 fueron:

| Prompt | Cambio aplicado en v2 | Motivo |
|---|---|---|
| P1 | Extensión expresada en **oraciones** + rango de palabras y estructura (origen → proceso → taza → invitación) | Los LLM controlan mejor la cantidad de oraciones que la de palabras. |
| P2 | Reglas de formato de *hashtags* (CamelCase, sin tildes ni duplicados), prohibición de copiar oraciones de P1 y ganchos distintos | Evitar *hashtags* inválidos y variantes repetitivas. |
| P3 | 55-75 palabras en frases cortas; exclusiones solo en el *negative prompt*; bolsa descrita como *"plain unbranded"* | Los modelos de difusión priorizan el inicio del prompt y "ven" los sustantivos aunque estén negados. |
| P4 | 5 o 6 oraciones, 60-80 palabras | Ajustar la duración real del audio a ~30 s. |
| P5 | Contexto de uso explícito ("se publica tal cual") y definición estricta de dato inventado | La rúbrica v1 no penalizaba textos con varias opciones ni métodos no incluidos en la ficha. |

La tabla compara el cumplimiento de restricciones entre ambas iteraciones con los mismos controles automáticos.

In [ ]:
ruta_it1 = DIR_OUT / "iteracion_1" / "resultados_pipeline.json"
if ruta_it1.exists():
    it1 = json.loads(ruta_it1.read_text(encoding="utf-8"))
    c1, c2 = evaluar_controles(it1), controles
    comparacion = pd.DataFrame({"Iteración 1 (%)": (c1.mean() * 100).round(0),
                                "Iteración 2 (%)": (c2.mean() * 100).round(0)})
    comparacion.loc["TOTAL"] = [round(c1.values.mean() * 100, 1), round(c2.values.mean() * 100, 1)]
    display(comparacion)
else:
    print("No se encontró la iteración 1.")

#### 6.7.d Rúbrica P0 vs. P1 (*LLM-as-judge*)
Para reducir el sesgo de posición, el texto optimizado se presenta como "A" en un lote y como "B" en otro.

In [ ]:
eval_filas = []
for i, (lid, r) in enumerate(RESULTADOS.items()):
    optim, base = r["p1_ficha"]["descripcion_extendida"], r["p0_baseline"]
    optim_es_a = (i % 2 == 0)
    texto_a, texto_b = (optim, base) if optim_es_a else (base, optim)
    ev = a_json(llm.generar(f"{lid}/p5", P5_RUBRICA.format(ficha_tecnica=ficha_a_texto(r["lote"]),
                                                          texto_a=texto_a, texto_b=texto_b),
                            temperatura=0, json_mode=True))
    for etiqueta, origen in (("A", "P1 optimizado" if optim_es_a else "P0 ingenuo"),
                             ("B", "P0 ingenuo" if optim_es_a else "P1 optimizado")):
        d = ev[etiqueta]
        eval_filas.append({"lote": lid, "prompt": origen,
                           **{k: d[k] for k in ("fidelidad", "claridad", "tono_marca", "persuasion", "uso_directo")},
                           "datos_inventados": len(d.get("datos_inventados", []))})
    print(f"{lid}: {ev.get('comentario', '')}")

evaluacion = pd.DataFrame(eval_filas)
criterios = ["fidelidad", "claridad", "tono_marca", "persuasion", "uso_directo"]
evaluacion["promedio"] = evaluacion[criterios].mean(axis=1).round(2)
display(evaluacion)
resumen_eval = evaluacion.groupby("prompt")[criterios + ["promedio", "datos_inventados"]].mean().round(2)
display(resumen_eval)

In [ ]:
import matplotlib.pyplot as plt
ax = resumen_eval[criterios].T.plot(kind="bar", figsize=(8, 4), color=["#c0392b", "#27ae60"], rot=0)
ax.set_ylim(0, 5.5); ax.set_ylabel("Puntaje (1-5)")
ax.set_title("Rúbrica P5: prompt ingenuo (P0) vs. prompt optimizado (P1)")
ax.legend(title="")
plt.tight_layout(); plt.show()

#### 6.7.e Uso de recursos (tokens, llamadas y latencia)
Permite dimensionar la viabilidad dentro de la capa gratuita.

In [ ]:
uso = pd.DataFrame(REGISTRO_USO)
display(uso)
print("Modelo utilizado:", ", ".join(sorted(set(uso["modelo"].dropna()))))
print("Llamadas registradas:", len(uso), "| por origen:", uso["origen"].value_counts().to_dict() if "origen" in uso else "")
if "tokens_total" in uso:
    print("Tokens totales:", int(uso["tokens_total"].fillna(0).sum()),
          "| promedio por llamada:", int(uso["tokens_total"].fillna(0).mean()))
    print("Costo en capa gratuita: $0")

### 6.8 Interfaz de usuario con Gradio *(contenido adicional)*

Permite que el personal de la cafetería ingrese un lote nuevo sin tocar código y obtenga el kit completo: descripción, *copy*, prompt para NightCafe y audio. Requiere `GEMINI_API_KEY` (en modo caché muestra un aviso). En GitHub la interfaz no se renderiza; se incluye una captura en `images/ui_gradio.png`.

In [ ]:
LANZAR_UI = True

def kit_desde_ui(pais, region, variedad, proceso, altitud, notas, acidez_cuerpo, puntaje, tueste, metodo, precio):
    if MODO != "api":
        return ("⚠️ Modo caché: configura GEMINI_API_KEY para generar contenido nuevo.", "", "", None)
    lote = {"id": "ui", "pais": pais, "region": region, "productor": "", "variedad": variedad,
            "proceso": proceso, "altitud": altitud, "notas": notas, "acidez_cuerpo": acidez_cuerpo,
            "puntaje_sca": puntaje, "tueste": tueste, "metodo_sugerido": metodo, "precio": precio}
    lote = {k: v for k, v in lote.items() if v not in ("", None)}
    prefijo = "ui_" + hashlib.md5(json.dumps(lote, sort_keys=True).encode()).hexdigest()[:8]
    r = ejecutar_pipeline(lote, prefijo=prefijo, guardar=False)
    f = r["p1_ficha"]
    ficha_md = (f"### {f['nombre_comercial']}\n**Carta:** {f['descripcion_menu']}\n\n{f['descripcion_extendida']}\n\n"
                + "\n".join(f"- **{n['nota']}** → {n['analogia']}" for n in f["notas_para_cliente"])
                + f"\n\n**Maridaje:** {f['maridaje']}")
    copy_md = "\n\n---\n\n".join(f"**{v['enfoque'].upper()}** — {v['gancho']}\n\n{v['cuerpo']}\n\n*{v['llamado_a_la_accion']}*\n\n{' '.join(v['hashtags'])}"
                                 for v in r["p2_copy"]["variantes"])
    prompt_img = f"{r['p3_imagen']['prompt_imagen']}\n\nNEGATIVE PROMPT: {r['p3_imagen']['prompt_negativo']}"
    ruta_audio = texto_a_audio(r["p4_guion"], DIR_AUDIO / f"{prefijo}.mp3")
    return ficha_md, copy_md, prompt_img, str(ruta_audio)

if LANZAR_UI:
    import gradio as gr
    ej = LOTES[1]
    with gr.Blocks(title="CafePrompt") as demo:
        gr.Markdown("# ☕ CafePrompt\nIngresa la ficha técnica del lote y obtén el kit de contenido.")
        with gr.Row():
            with gr.Column():
                c = [gr.Textbox(label="País", value=ej["pais"]), gr.Textbox(label="Región", value=ej["region"]),
                     gr.Textbox(label="Variedad", value=ej["variedad"]), gr.Textbox(label="Proceso", value=ej["proceso"]),
                     gr.Textbox(label="Altitud", value=ej["altitud"]), gr.Textbox(label="Notas de cata", value=ej["notas"]),
                     gr.Textbox(label="Acidez y cuerpo", value=ej["acidez_cuerpo"]),
                     gr.Number(label="Puntaje SCA", value=ej["puntaje_sca"]), gr.Textbox(label="Tueste", value=ej["tueste"]),
                     gr.Textbox(label="Método sugerido", value=ej["metodo_sugerido"]), gr.Textbox(label="Precio", value=ej["precio"])]
                boton = gr.Button("Generar kit", variant="primary")
            with gr.Column():
                o_ficha = gr.Markdown(label="Ficha sensorial")
                o_audio = gr.Audio(label="Locución", type="filepath")
                o_prompt = gr.Textbox(label="Prompt para NightCafe", lines=6)
        o_copy = gr.Markdown(label="Copy Instagram")
        boton.click(kit_desde_ui, inputs=c, outputs=[o_ficha, o_copy, o_prompt, o_audio])
    demo.launch(share=False, inline=True, debug=False)

### 6.9 Persistencia de resultados
Se guarda la caché con las respuestas reales (para ejecutar el notebook sin clave) y un resumen de los artefactos generados.

In [ ]:
CACHE_PATH.write_text(json.dumps(CACHE, ensure_ascii=False, indent=2), encoding="utf-8")
(DIR_OUT / "resultados_pipeline.json").write_text(json.dumps(RESULTADOS, ensure_ascii=False, indent=2), encoding="utf-8")
print("Archivos en outputs/:")
for p in sorted(DIR_OUT.rglob("*")):
    if p.is_file():
        print(f"  {p}  ({p.stat().st_size/1024:.1f} KB)")

# En Colab con API, se descargan los artefactos para versionarlos en el repositorio
if EN_COLAB and MODO == "api":
    import shutil
    from google.colab import files
    shutil.make_archive("artefactos_cafeprompt", "zip", DIR_OUT)
    files.download("artefactos_cafeprompt.zip")

## 7. Resultados

_Pendiente de completar tras la ejecución._

## 8. Conclusiones

_Pendiente de completar tras la ejecución._

## 9. Referencias

1. Brown, T. et al. (2020). *Language Models are Few-Shot Learners*. NeurIPS. https://arxiv.org/abs/2005.14165
2. Wei, J. et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*. https://arxiv.org/abs/2201.11903
3. White, J. et al. (2023). *A Prompt Pattern Catalog to Enhance Prompt Engineering with ChatGPT*. https://arxiv.org/abs/2302.11382
4. Zheng, L. et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*. https://arxiv.org/abs/2306.05685
5. Liu, V. & Chilton, L. (2022). *Design Guidelines for Prompt Engineering Text-to-Image Generative Models*. CHI '22. https://arxiv.org/abs/2109.06977
6. Oppenlaender, J. (2023). *A Taxonomy of Prompt Modifiers for Text-To-Image Generation*. https://arxiv.org/abs/2204.13988
7. OpenAI. *Prompt engineering guide*. https://platform.openai.com/docs/guides/prompt-engineering
8. Google. *Prompt design strategies — Gemini API*. https://ai.google.dev/gemini-api/docs/prompting-strategies
9. Google. *Gemini models* y *Rate limits*. https://ai.google.dev/gemini-api/docs/models · https://ai.google.dev/gemini-api/docs/rate-limits
10. Google. *Google Gen AI Python SDK*. https://github.com/googleapis/python-genai
11. NightCafe Studio. https://creator.nightcafe.studio
12. gTTS — Google Text-to-Speech. https://gtts.readthedocs.io
13. Gradio. *Documentation*. https://www.gradio.app/docs
14. Specialty Coffee Association & World Coffee Research (2016). *Coffee Taster's Flavor Wheel*. https://sca.coffee/research/coffee-tasters-flavor-wheel
15. Specialty Coffee Association. *Cupping Protocols*. https://sca.coffee/research/protocols-best-practices